# Car Classification
### Dataset: https://www.kaggle.com/datasets/eduardo4jesus/stanford-cars-dataset

#### Library Imports

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from scipy.io import loadmat
from sklearn.model_selection import train_test_split

%matplotlib inline
sns.set_theme(style="white")

##### Load Data

In [ ]:
base = "../data/carclassificationdataset1/car_devkit/devkit"

#Load class ID and names
meta = loadmat(f"{base}/cars_meta.mat")
class_names = [c[0] for c in meta['class_names'][0]]
print(f"Number of classes: {len(class_names)}")
print(class_names[:5])


In [ ]:
#Annotation Inspection
train_annos = loadmat(f"{base}/cars_train_annos.mat")
print(train_annos['annotations'][0][0])

In [ ]:
data = []
for anno in train_annos['annotations'][0]:
    data.append({
        'filename': anno[5][0],
        'x1': anno[0][0][0],
        'y1': anno[1][0][0],
        'x2': anno[2][0][0],
        'y2': anno[3][0][0],
        'class_id': anno[4][0][0],
    })

df_train = pd.DataFrame(data)
df_train.head()

In [ ]:
df_train['class_name'] = df_train['class_id'].apply(lambda cid: class_names[cid - 1])
df_train.head()

In [ ]:
test_annos = loadmat(f"{base}/cars_test_annos.mat")
print(test_annos['annotations'][0][0])

##### Data Exploration

In [ ]:
class_counts = df_train['class_name'].value_counts()
print(class_counts.describe())
print("\nClasses with fewest images:")
print(class_counts.tail(10))
print('\nClasses with the most images:')
print(class_counts.head(10))

In [ ]:
sizes = []
for fname in df_train['filename'].sample(500, random_state=42):
    img_path = f"../data/carclassificationdataset1/cars_train/cars_train/{fname}"
    with Image.open(img_path) as img:
        sizes.append(img.size)

df_sizes = pd.DataFrame(sizes, columns=['width', 'height'])
df_sizes.describe()

In [ ]:
sample_class = df_train['class_name'].iloc[0]
sample_imgs = df_train[df_train['class_name'] == sample_class].head(6)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (_, row) in zip(axes.flatten(), sample_imgs.iterrows()):
    img_path = f"../data/carclassificationdataset1/cars_train/cars_train/{row['filename']}"
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(row['filename'])
    ax.axis('off')

plt.suptitle(sample_class)
plt.tight_layout()
plt.show()

##### Run Split (70/15/15)

In [ ]:
train_df, temp_df = train_test_split(df_train, test_size=0.3, stratify=df_train['class_id'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['class_id'], random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

##### Build Dataset class for resizing and cropping

In [ ]:
class CarDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{self.img_dir}/{row['filename']}").convert("RGB")
        img = img.crop((row['x1'], row['y1'], row['x2'], row['y2']))
        if self.transform:
            img = self.transform(img)
        label = row['class_id'] - 1  # 0-indexed for PyTorch
        return img, label

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
img_dir = "../data/carclassificationdataset1/cars_train/cars_train"
train_dataset = CarDataset(train_df, img_dir, transform=transform)
val_dataset = CarDataset(val_df, img_dir, transform=transform)
test_dataset = CarDataset(test_df, img_dir, transform=transform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
images, labels = next(iter(train_loader))
print(f"Batch of images shape: {images.shape}")
print(f"Batch of labels shape: {labels.shape}")
print(labels.min(), labels.max())

In [ ]:
import torchvision.models as models
import torch.nn as nn

model = models.resnet50(weights='IMAGENET1K_V2')
model.fc = nn.Linear(model.fc.in_features, 196)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4)

##### Run on GPU rather than CPU with CUDA enabled

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(device)

##### Training Loop
Run on 1 epoch first before commiting to 10

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    # --- Training phase ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # --- Validation phase ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

In [ ]:
model.eval()
test_correct = 0
test_total = 0
test_loss = 0.0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        test_correct += (predicted == labels).sum().item()
        test_total += labels.size(0)

test_loss = test_loss / test_total
test_acc = test_correct / test_total

print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

##### Data Augmentation
apply random transformations to training images each epoch

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Val/test not augmented
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
#Rebuild datasets and dataloaders with new transforms
train_dataset = CarDataset(train_df, img_dir, transform=train_transform)
val_dataset   = CarDataset(val_df, img_dir, transform=eval_transform)
test_dataset  = CarDataset(test_df, img_dir, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
model = models.resnet50(weights='IMAGENET1K_V2')
model.fc = nn.Linear(model.fc.in_features, 196)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  # fresh optimizer too

In [ ]:
num_epochs = 20
best_val_acc = 0.0

for epoch in range(num_epochs):
    # --- Training phase ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # --- Validation phase ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "../models/classifier/resnet50_stanford_cars_best.pth")
        print(f"  → New best model saved (Val Acc: {val_acc:.4f})")

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

In [ ]:
model.load_state_dict(torch.load("../models/classifier/resnet50_stanford_cars_best.pth"))
model.eval()

test_correct = 0
test_total = 0
test_loss = 0.0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        test_correct += (predicted == labels).sum().item()
        test_total += labels.size(0)

test_loss = test_loss / test_total
test_acc = test_correct / test_total

print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")